# 03 · Train MIL models (Week 4)

Trains **6 MIL models** (maxpool, meanpool, ABMIL, DSMIL, CLAM-SB, TransMIL) on **4 encoders' features**
(ViT-S/16 Lunit, CTransPath, Phikon-v2, PLIP) × **5 folds** = 120 runs, then scores the **locked 108-slide test set**
with 95% confidence intervals. Features are loaded onto the GPUs once; both GPUs work in parallel.

### Settings (right panel)
1. **Add Input → Your Work → Notebooks → `notebook4297f7f927`** (the Week 2 features). The competition data is NOT needed.
2. **Session options → Accelerator → GPU T4 x2**
3. **Session options → Internet → On**

### Step A · smoke test (~15 min, interactive)
`SMOKE_TEST = True` → **Run All**. It trains 3 models on 1 fold of ViT-S/16 for 5 epochs and prints a small results table.

### Step B · full run (background)
`SMOKE_TEST = False` → **Save Version → Save & Run All (Commit)**.

### Step C · only if Step B stopped early
Add Input → Your Work → Notebooks → **this notebook** (its output), then commit again. Finished runs are copied, not redone.

### Seeds run (3 seeds) · `SEEDS = '0,1,2'` (default)
Attach the output of the earlier seed-0 training run (Add Input → Your Work → Notebooks → that notebook). Its 120 seed-0 runs
are copied; only seeds 1 and 2 are trained (240 runs, ~3.5 h). Results then use all 15 models (3 seeds × 5 folds) per
combination, and cell 5b runs the paired significance tests on the test set (108 slides) and on the CV predictions (430 slides).

In [ ]:
# 1 · libraries + start clock
import time; T0 = time.time()
!pip install -q nystrom-attention einops
import torch; print('torch', torch.__version__, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
import os; os.makedirs('/tmp/code/mil', exist_ok=True); os.makedirs('/tmp/code/splits_v2', exist_ok=True)

In [ ]:
%%writefile /tmp/code/mil/__init__.py
"""MIL aggregators, training and evaluation for UBC-OCEAN subtype classification."""

In [ ]:
%%writefile /tmp/code/mil/models.py
"""Slide-level MIL aggregators.

Every model takes one bag of patch features ``x`` with shape (N, D) and returns a dict:
    logits    (C,)      class scores for the slide
    attn      (N,)      per-patch importance (for heatmaps), or None
    aux_loss  scalar    extra training loss (DSMIL instance loss, CLAM clustering loss), or None

Optional extra key:
    loss_logits (C,)    logits to use for the main training loss instead of ``logits`` (DSMIL)

Models with an auxiliary loss define ``aux_weight``: total = (1 - w) * main + w * aux.
Models that need the slide label for their auxiliary loss (DSMIL, CLAM) accept ``label=``;
the rest ignore it. Build models with ``build_model(name, in_dim, num_classes)``.
"""
import math

import torch
import torch.nn.functional as F
from torch import nn


class PoolMIL(nn.Module):
    """Per-patch MLP, then max or mean pooling, then a linear classifier.

    Layer names match the old capstone checkpoints (encoder.0, encoder.3, classifier),
    so those weights can still be loaded.
    """

    def __init__(self, in_dim, num_classes, pool='max', dropout=0.25):
        super().__init__()
        assert pool in ('max', 'mean')
        self.pool = pool
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x, label=None):
        h = self.encoder(x)                                   # (N, 128)
        if self.pool == 'max':
            pooled, idx = h.max(dim=0)
            # importance = how many of the 128 features each patch "won"
            attn = torch.bincount(idx, minlength=len(x)).float() / h.shape[1]
        else:
            pooled, attn = h.mean(dim=0), None
        return dict(logits=self.classifier(pooled), attn=attn, aux_loss=None)


class GatedAttention(nn.Module):
    """Gated attention scores (Ilse et al. 2018). Returns raw scores (N, n_out)."""

    def __init__(self, dim, hidden, n_out=1, dropout=0.25):
        super().__init__()
        self.a = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Dropout(dropout))
        self.b = nn.Sequential(nn.Linear(dim, hidden), nn.Sigmoid(), nn.Dropout(dropout))
        self.c = nn.Linear(hidden, n_out)

    def forward(self, h):
        return self.c(self.a(h) * self.b(h))


class ABMIL(nn.Module):
    """Gated attention MIL (Ilse et al. 2018).

    Fixes old bug B1: attention is normalized over PATCHES (dim 0 of an (N, 1) tensor),
    not over a size-1 batch axis.
    """

    def __init__(self, in_dim, num_classes, hidden=256, dropout=0.25):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden, hidden // 2, 1, dropout)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, x, label=None):
        h = self.fc(x)                                        # (N, H)
        a = torch.softmax(self.attention(h).squeeze(1), 0)    # (N,)
        m = a @ h                                             # (H,)
        return dict(logits=self.classifier(m), attn=a, aux_loss=None)


class DSMIL(nn.Module):
    """Dual-stream MIL (Li, Li & Eliceiri, CVPR 2021). Cleaned port of the capstone version.

    Stream 1 scores each patch; the top patch per class is the "critical instance".
    Stream 2 attends to all patches by similarity to the critical instances.
    Prediction = mean of both streams; aux_loss = max-instance cross-entropy.
    """

    aux_weight = 0.5

    def __init__(self, in_dim, num_classes, hidden=128, dropout=0.25):
        super().__init__()
        self.i_classifier = nn.Linear(in_dim, num_classes)
        self.q = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.Tanh())
        self.v = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_dim, in_dim), nn.ReLU())
        self.b_classifier = nn.Conv1d(num_classes, num_classes, kernel_size=in_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, label=None):
        inst_logits = self.i_classifier(x)                    # (N, C)
        q, v = self.q(x), self.v(x)                           # (N, h), (N, D)
        crit = inst_logits.argmax(dim=0)                      # (C,) top patch per class
        attn = torch.softmax(q @ q[crit].T / math.sqrt(q.shape[1]), dim=0)  # (N, C)
        bag_feat = self.dropout(attn.T @ v)                   # (C, D)
        bag_logits = self.b_classifier(bag_feat.unsqueeze(0)).squeeze()      # (C,)
        max_inst = inst_logits.max(dim=0).values              # (C,)

        aux = None
        if label is not None:
            aux = F.cross_entropy(max_inst.unsqueeze(0), label.view(1))
        # Prediction = mean of both streams' probabilities (as log-probs, so softmax(logits) == probs).
        # Training loss = 0.5 * CE(bag_logits) + 0.5 * CE(max_inst), as in the original DSMIL.
        probs = 0.5 * (bag_logits.softmax(0) + max_inst.softmax(0))
        pred = probs.argmax()
        return dict(logits=probs.clamp_min(1e-8).log(), attn=attn[:, pred], aux_loss=aux,
                    loss_logits=bag_logits)


class CLAM_SB(nn.Module):
    """CLAM single-branch (Lu et al., Nat. Biomed. Eng. 2021).

    Gated attention pooling plus an instance-level clustering loss: the k most-attended
    patches are pushed to "positive" for the true class and "negative" for other classes;
    the k least-attended are "negative" for the true class.
    """

    aux_weight = 0.3   # official CLAM: 0.7 * bag loss + 0.3 * instance loss

    def __init__(self, in_dim, num_classes, hidden=512, attn_hidden=256, dropout=0.25, k_sample=8):
        super().__init__()
        self.k_sample = k_sample
        self.num_classes = num_classes
        self.fc = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden, attn_hidden, 1, dropout)
        self.classifier = nn.Linear(hidden, num_classes)
        self.instance_classifiers = nn.ModuleList(nn.Linear(hidden, 2) for _ in range(num_classes))

    def _inst_loss(self, a, h, c, in_class):
        k = min(self.k_sample, len(h) // 2 if in_class else len(h))
        if k < 1:
            return h.new_zeros(())
        top = h[torch.topk(a, k).indices]
        if in_class:
            bottom = h[torch.topk(-a, k).indices]
            feats = torch.cat([top, bottom])
            targets = torch.cat([torch.ones(k), torch.zeros(k)]).long().to(h.device)
        else:
            feats, targets = top, torch.zeros(k, dtype=torch.long, device=h.device)
        return F.cross_entropy(self.instance_classifiers[c](feats), targets)

    def forward(self, x, label=None):
        h = self.fc(x)                                        # (N, H)
        a = torch.softmax(self.attention(h).squeeze(1), 0)    # (N,)
        logits = self.classifier(a @ h)

        aux = None
        if label is not None:
            y = int(label)
            losses = [self._inst_loss(a, h, c, in_class=(c == y)) for c in range(self.num_classes)]
            aux = torch.stack(losses).mean()
        return dict(logits=logits, attn=a, aux_loss=aux)


class PPEG(nn.Module):
    """Pyramid position encoding generator from TransMIL."""

    def __init__(self, dim):
        super().__init__()
        self.c7 = nn.Conv2d(dim, dim, 7, 1, 3, groups=dim)
        self.c5 = nn.Conv2d(dim, dim, 5, 1, 2, groups=dim)
        self.c3 = nn.Conv2d(dim, dim, 3, 1, 1, groups=dim)

    def forward(self, x, side):
        cls, feat = x[:, :1], x[:, 1:]
        grid = feat.transpose(1, 2).reshape(x.shape[0], -1, side, side)
        grid = grid + self.c7(grid) + self.c5(grid) + self.c3(grid)
        return torch.cat([cls, grid.flatten(2).transpose(1, 2)], dim=1)


class TransLayer(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        from nystrom_attention import NystromAttention
        self.norm = nn.LayerNorm(dim)
        self.attn = NystromAttention(dim=dim, dim_head=dim // 8, heads=8, num_landmarks=dim // 2,
                                     pinv_iterations=6, residual=True, dropout=dropout)

    def forward(self, x):
        return x + self.attn(self.norm(x))

    @torch.no_grad()
    def cls_attention(self, x):
        """Exact softmax attention from token 0 (class token) to every token, mean over heads.

        O(N) memory, unlike asking NystromAttention for its full N×N matrix.
        """
        q, k, _ = self.attn.to_qkv(self.norm(x)).chunk(3, dim=-1)       # (1, L, heads*dh)
        heads = self.attn.heads
        q = q[0, :1].view(1, heads, -1).transpose(0, 1)                  # (heads, 1, dh)
        k = k[0].view(x.shape[1], heads, -1).transpose(0, 1)             # (heads, L, dh)
        scores = (q * self.attn.scale) @ k.transpose(1, 2)               # (heads, 1, L)
        return scores.softmax(-1).mean(0)[0]                             # (L,)


class TransMIL(nn.Module):
    """TransMIL (Shao et al., NeurIPS 2021): Nystrom self-attention between patches + PPEG.

    attn = attention from the class token to each patch in the last layer (mean over heads).
    """

    def __init__(self, in_dim, num_classes, hidden=512, dropout=0.25):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden))
        self.layer1 = TransLayer(hidden)
        self.ppeg = PPEG(hidden)
        self.layer2 = TransLayer(hidden)
        self.norm = nn.LayerNorm(hidden)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, x, label=None):
        n = len(x)
        h = self.fc(x).unsqueeze(0)                           # (1, N, H)
        side = math.ceil(math.sqrt(n))
        pad = side * side - n
        if pad:                                               # fill the square grid by repeating patches
            h = torch.cat([h, h[:, :pad]], dim=1)
        h = torch.cat([self.cls_token, h], dim=1)             # (1, 1 + side², H)
        h = self.layer1(h)
        h = self.ppeg(h, side)
        attn = None if self.training else self.layer2.cls_attention(h)[1:n + 1]   # (N,) real patches only
        h = self.layer2(h)
        logits = self.classifier(self.norm(h)[0, 0])
        return dict(logits=logits, attn=attn, aux_loss=None)


MODELS = {
    'maxpool': lambda d, c, **kw: PoolMIL(d, c, pool='max', **kw),
    'meanpool': lambda d, c, **kw: PoolMIL(d, c, pool='mean', **kw),
    'abmil': ABMIL,
    'dsmil': DSMIL,
    'clam_sb': CLAM_SB,
    'transmil': TransMIL,
}


def build_model(name, in_dim, num_classes, **kwargs):
    """kwargs: model hyperparameters, e.g. hidden=256, dropout=0.25 (None values are ignored)."""
    if name not in MODELS:
        raise ValueError(f'unknown model {name!r}; choose from {sorted(MODELS)}')
    return MODELS[name](in_dim, num_classes, **{k: v for k, v in kwargs.items() if v is not None})

In [ ]:
%%writefile /tmp/code/mil/data.py
"""Bags of patch features, one per slide.

Expected layout (written by the Week 2 extraction notebook):
    <features_dir>/<slide_id>.npy    (N, D) float16, one row per tissue tile
Split CSVs (from make_splits.py): file_name,label,is_tma
"""
import os

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

CLASSES = ['CC', 'EC', 'HGSC', 'LGSC', 'MC']


def read_split(path):
    return pd.read_csv(path, dtype={'file_name': str})


class BagDataset(Dataset):
    """Returns (features (N, D) float32, label (), slide_id str).

    max_patches: if set, a random subset of at most this many patches is drawn each time
    (training only — saves memory on huge slides and acts as augmentation).
    cache: False = read from disk every time; True = keep bags in RAM after first load;
           a dict {slide_id: array or torch tensor} = use these preloaded bags (e.g. already on the GPU,
           see preload_features), shared between datasets.
    """

    def __init__(self, df, features_dir, max_patches=None, cache=False, seed=0):
        self.df = df.reset_index(drop=True)
        self.features_dir = features_dir
        self.max_patches = max_patches
        self.cache = cache if isinstance(cache, dict) else ({} if cache else None)
        self.rng = np.random.default_rng(seed)
        missing = [s for s in self.df.file_name
                   if not (self.cache is not None and s in self.cache) and not os.path.exists(self._path(s))]
        if missing:
            raise FileNotFoundError(f'{len(missing)} slides have no features in {features_dir}, e.g. {missing[:3]}')

    def _path(self, slide_id):
        return os.path.join(self.features_dir, f'{slide_id}.npy')

    def _load(self, slide_id):
        if self.cache is not None and slide_id in self.cache:
            return self.cache[slide_id]
        x = np.load(self._path(slide_id))
        if self.cache is not None:
            self.cache[slide_id] = x
        return x

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        x = self._load(row.file_name)
        if self.max_patches and len(x) > self.max_patches:
            idx = np.sort(self.rng.choice(len(x), self.max_patches, replace=False))
            x = x[torch.from_numpy(idx).to(x.device)] if torch.is_tensor(x) else x[idx]
        x = x.float() if torch.is_tensor(x) else torch.from_numpy(np.asarray(x, dtype=np.float32))
        return x, torch.tensor(int(row.label)), row.file_name

    def feature_dim(self):
        return self._load(self.df.file_name.iloc[0]).shape[1]


def preload_features(features_dir, slide_ids, device='cpu'):
    """Load every bag once as a float16 tensor on `device` (e.g. 'cuda:0'). Returns {slide_id: tensor}."""
    return {s: torch.from_numpy(np.load(os.path.join(features_dir, f'{s}.npy'))).to(device) for s in slide_ids}

In [ ]:
%%writefile /tmp/code/mil/metrics.py
"""Slide-level metrics and bootstrap confidence intervals.

All functions take a predictions DataFrame with columns:
    image_id, label, pred, prob_CC, prob_EC, prob_HGSC, prob_LGSC, prob_MC  (+ optional is_tma)
"""
import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, roc_auc_score

from .data import CLASSES

PROB_COLS = [f'prob_{c}' for c in CLASSES]
LABELS = list(range(len(CLASSES)))


def macro_auc(y, probs):
    """One-vs-rest AUROC averaged over classes that have both positives and negatives."""
    aucs = []
    for c in LABELS:
        pos = (y == c)
        if 0 < pos.sum() < len(y):
            aucs.append(roc_auc_score(pos, probs[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan')


def compute_metrics(df):
    y, p = df['label'].to_numpy(), df['pred'].to_numpy()
    probs = df[PROB_COLS].to_numpy()
    out = dict(
        n=len(df),
        acc=accuracy_score(y, p),
        bacc=balanced_accuracy_score(y, p) if len(set(y)) > 1 else float('nan'),
        macro_f1=f1_score(y, p, labels=LABELS, average='macro', zero_division=0),
        macro_auc=macro_auc(y, probs),
    )
    recalls = recall_score(y, p, labels=LABELS, average=None, zero_division=0)
    out.update({f'recall_{c}': float(r) for c, r in zip(CLASSES, recalls)})
    return out


def bootstrap_ci(df, metrics=('acc', 'bacc', 'macro_f1', 'macro_auc'), n_boot=1000, seed=0, alpha=0.05):
    """Percentile bootstrap over slides. Returns {metric: (low, high)}."""
    rng = np.random.default_rng(seed)
    values = {m: [] for m in metrics}
    for _ in range(n_boot):
        sample = df.iloc[rng.integers(0, len(df), len(df))]
        res = compute_metrics(sample)
        for m in metrics:
            values[m].append(res[m])
    return {m: (float(np.nanpercentile(v, 100 * alpha / 2)), float(np.nanpercentile(v, 100 * (1 - alpha / 2))))
            for m, v in values.items()}


def ensemble(pred_dfs):
    """Average class probabilities of several models (e.g. the 5 fold models) per slide."""
    base = pred_dfs[0][['image_id', 'label'] + (['is_tma'] if 'is_tma' in pred_dfs[0] else [])].copy()
    probs = np.mean([d.set_index('image_id').loc[base.image_id, PROB_COLS].to_numpy() for d in pred_dfs], axis=0)
    base[PROB_COLS] = probs
    base['pred'] = probs.argmax(1)
    return base

In [ ]:
%%writefile /tmp/code/mil/train.py
"""Train one MIL model on one (or all) CV folds, then predict the locked test set.

Model selection uses the fold's VALIDATION set only. The test set is predicted once,
with the selected checkpoint, and never influences training (fixes old issue M1).

Example:
    python -m mil.train --features features/uni --splits splits_v2 --model abmil --fold all --out runs/uni/abmil/seed0

Outputs per fold in <out>/fold<k>/:
    config.json  history.csv  best.pt  val_preds.csv  test_preds.csv
"""
import argparse
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from .data import CLASSES, BagDataset, read_split
from .metrics import compute_metrics
from .models import build_model


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def bag_loader(ds, shuffle, seed):
    g = torch.Generator().manual_seed(seed)
    # batch_size=1 bag; collate keeps (x, y, id) without stacking
    return DataLoader(ds, batch_size=1, shuffle=shuffle, generator=g, num_workers=0,
                      collate_fn=lambda b: b[0])


def loss_fn(model, out, y, class_weights):
    main = F.cross_entropy(out.get('loss_logits', out['logits']).unsqueeze(0), y.view(1), weight=class_weights)
    if out['aux_loss'] is None:
        return main
    w = getattr(model, 'aux_weight', 0.0)
    return (1 - w) * main + w * out['aux_loss']


@torch.no_grad()
def predict(model, loader, device, df, class_weights=None):
    """Returns (preds DataFrame, mean loss)."""
    model.eval()
    rows, total = [], 0.0
    for x, y, sid in loader:
        x, y = x.to(device), y.to(device)
        out = model(x, label=y)
        total += loss_fn(model, out, y, class_weights).item()
        probs = out['logits'].float().softmax(0).cpu().numpy()
        rows.append(dict(image_id=sid, label=int(y), pred=int(probs.argmax()),
                         **{f'prob_{c}': float(p) for c, p in zip(CLASSES, probs)}))
    preds = pd.DataFrame(rows).merge(df[['file_name', 'is_tma']].rename(columns={'file_name': 'image_id'}),
                                     on='image_id', how='left')
    return preds, total / max(1, len(loader))


def train_fold(args, fold, device, feature_cache=None):
    """feature_cache: optional {slide_id: tensor} from data.preload_features, shared across runs."""
    set_seed(args.seed + fold)
    out_dir = os.path.join(args.out, f'fold{fold}')
    os.makedirs(out_dir, exist_ok=True)

    train_df = read_split(os.path.join(args.splits, 'train', f'{fold}.csv'))
    val_df = read_split(os.path.join(args.splits, 'val', f'{fold}.csv'))
    test_df = read_split(os.path.join(args.splits, 'test.csv'))

    cache = feature_cache if feature_cache is not None else args.cache
    train_ds = BagDataset(train_df, args.features, max_patches=args.max_patches, cache=cache, seed=args.seed + fold)
    val_ds = BagDataset(val_df, args.features, cache=cache)
    test_ds = BagDataset(test_df, args.features, cache=feature_cache if feature_cache is not None else False)
    in_dim = train_ds.feature_dim()

    model_kwargs = {k: getattr(args, k, None) for k in ('hidden', 'dropout') if getattr(args, k, None) is not None}
    model = build_model(args.model, in_dim, len(CLASSES), **model_kwargs).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)

    class_weights = None
    if args.class_weights:
        counts = np.bincount(train_df.label, minlength=len(CLASSES))
        class_weights = torch.tensor(len(train_df) / (len(CLASSES) * np.maximum(counts, 1)), dtype=torch.float32, device=device)

    with open(os.path.join(out_dir, 'config.json'), 'w') as f:
        json.dump({**vars(args), 'fold': fold, 'in_dim': in_dim, 'device': str(device)}, f, indent=2)

    train_loader = bag_loader(train_ds, shuffle=True, seed=args.seed + fold)
    val_loader = bag_loader(val_ds, shuffle=False, seed=0)

    best, best_epoch, bad_epochs, history = None, 0, 0, []
    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        model.train()
        opt.zero_grad()
        train_loss = 0.0
        for i, (x, y, _) in enumerate(train_loader, 1):
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model, model(x, label=y), y, class_weights) / args.accum
            loss.backward()
            train_loss += loss.item() * args.accum
            if i % args.accum == 0 or i == len(train_loader):
                opt.step()
                opt.zero_grad()

        val_preds, val_loss = predict(model, val_loader, device, val_df, class_weights)
        m = compute_metrics(val_preds)
        history.append(dict(epoch=epoch, train_loss=train_loss / len(train_loader), val_loss=val_loss,
                            val_bacc=m['bacc'], val_f1=m['macro_f1'], val_auc=m['macro_auc'], sec=time.time() - t0))
        score = -val_loss if args.select == 'loss' else m['bacc']
        improved = best is None or score > best
        if improved:
            best, best_epoch, bad_epochs = score, epoch, 0
            torch.save({'model': model.state_dict(), 'epoch': epoch, 'model_name': args.model, 'model_kwargs': model_kwargs,
                        'in_dim': in_dim, 'classes': CLASSES}, os.path.join(out_dir, 'best.pt'))
        elif epoch >= args.min_epochs:
            bad_epochs += 1

        h = history[-1]
        print(f'fold {fold} ep {epoch:3d} | train {h["train_loss"]:.3f} | val loss {val_loss:.3f} '
              f'bacc {m["bacc"]:.3f} f1 {m["macro_f1"]:.3f} | {h["sec"]:.0f}s{" *" if improved else ""}', flush=True)
        if bad_epochs >= args.patience:
            break

    pd.DataFrame(history).to_csv(os.path.join(out_dir, 'history.csv'), index=False)

    # reload the selected checkpoint, then predict val (for OOF metrics) and the locked test set
    model.load_state_dict(torch.load(os.path.join(out_dir, 'best.pt'), map_location=device)['model'])
    val_preds, _ = predict(model, val_loader, device, val_df)
    test_preds, _ = predict(model, bag_loader(test_ds, shuffle=False, seed=0), device, test_df)
    val_preds.to_csv(os.path.join(out_dir, 'val_preds.csv'), index=False)
    test_preds.to_csv(os.path.join(out_dir, 'test_preds.csv'), index=False)
    print(f'fold {fold} done: best epoch {best_epoch}, val bacc {compute_metrics(val_preds)["bacc"]:.3f}')


def main():
    p = argparse.ArgumentParser(description='Train a MIL aggregator on precomputed patch features')
    p.add_argument('--features', required=True, help='folder with <slide_id>.npy feature files')
    p.add_argument('--splits', required=True, help='folder from make_splits.py (train/, val/, test.csv)')
    p.add_argument('--model', required=True, help='maxpool, meanpool, abmil, dsmil, clam_sb, transmil')
    p.add_argument('--fold', default='all', help="fold number (1-5) or 'all'")
    p.add_argument('--out', required=True)
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--epochs', type=int, default=50)
    p.add_argument('--min_epochs', type=int, default=10, help='early stopping cannot trigger before this')
    p.add_argument('--patience', type=int, default=10)
    p.add_argument('--lr', type=float, default=2e-4)
    p.add_argument('--wd', type=float, default=1e-4)
    p.add_argument('--accum', type=int, default=1, help='gradient accumulation over this many bags')
    p.add_argument('--max_patches', type=int, default=8000, help='random patch cap per training bag (0 = no cap)')
    p.add_argument('--select', choices=['loss', 'bacc'], default='loss', help='validation criterion for the best epoch')
    p.add_argument('--class_weights', action='store_true', help='inverse-frequency class weights in the loss')
    p.add_argument('--hidden', type=int, default=None, help='model hidden size (default: model default)')
    p.add_argument('--dropout', type=float, default=None, help='model dropout (default: model default)')
    p.add_argument('--cache', action='store_true', help='keep features in RAM')
    p.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')
    args = p.parse_args()
    args.max_patches = args.max_patches or None

    folds = range(1, 6) if args.fold == 'all' else [int(args.fold)]
    for k in folds:
        train_fold(args, k, torch.device(args.device))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /tmp/code/mil/evaluate.py
"""Summarize finished runs into one results table.

For each run folder (containing fold1/ ... fold5/ from mil.train):
  * CV  = pooled out-of-fold validation predictions (used for model selection only)
  * Test = the 5 fold models' probabilities averaged per slide on the locked test set,
           with 95% bootstrap confidence intervals. This is the number to report.

Example:
    python -m mil.evaluate runs/uni/*/seed0 --out results/summary.csv
"""
import argparse
import glob
import json
import os

import numpy as np
import pandas as pd

from .metrics import bootstrap_ci, compute_metrics, ensemble

MAIN = ['acc', 'bacc', 'macro_f1', 'macro_auc']


def summarize_run(run_dir, n_boot=1000):
    folds = sorted(glob.glob(os.path.join(run_dir, 'fold*')))
    folds = [f for f in folds if os.path.exists(os.path.join(f, 'test_preds.csv'))]
    if not folds:
        raise FileNotFoundError(f'no finished folds in {run_dir}')

    val = pd.concat([pd.read_csv(os.path.join(f, 'val_preds.csv'), dtype={'image_id': str}) for f in folds])
    tests = [pd.read_csv(os.path.join(f, 'test_preds.csv'), dtype={'image_id': str}) for f in folds]
    test_ens = ensemble(tests)
    test_ens.to_csv(os.path.join(run_dir, 'test_preds_ensemble.csv'), index=False)

    cv = compute_metrics(val)
    te = compute_metrics(test_ens)
    ci = bootstrap_ci(test_ens, MAIN, n_boot=n_boot) if n_boot else {m: (np.nan, np.nan) for m in MAIN}
    per_fold = pd.DataFrame([compute_metrics(t) for t in tests])

    row = {'run': run_dir, 'n_folds': len(folds), 'n_test': te['n']}
    row.update({f'cv_{m}': cv[m] for m in MAIN})
    for m in MAIN:
        row[f'test_{m}'] = te[m]
        row[f'test_{m}_lo'], row[f'test_{m}_hi'] = ci[m]
        row[f'foldtest_{m}_mean'] = per_fold[m].mean()
        row[f'foldtest_{m}_std'] = per_fold[m].std()
    row.update({f'test_{k}': v for k, v in te.items() if k.startswith('recall_')})
    if 'is_tma' in test_ens:
        for name, sub in [('wsi', test_ens[test_ens.is_tma == 0]), ('tma', test_ens[test_ens.is_tma == 1])]:
            if len(sub):
                s = compute_metrics(sub)
                row[f'test_{name}_n'], row[f'test_{name}_acc'], row[f'test_{name}_bacc'] = s['n'], s['acc'], s['bacc']

    with open(os.path.join(run_dir, 'summary.json'), 'w') as f:
        json.dump(row, f, indent=2, default=float)
    return row


def summarize_config(config_dir, n_boot=1000):
    """Aggregate all seeds of one encoder+model folder (<config_dir>/seed*/fold*).

    Returns mean ± std across seeds of each seed's test metrics, plus the SEED ENSEMBLE:
    probabilities of all seeds x folds averaged per test slide (saved as test_preds_seedensemble.csv), with 95% CI.
    """
    seed_dirs = sorted(d for d in glob.glob(os.path.join(config_dir, 'seed*'))
                       if glob.glob(os.path.join(d, 'fold*', 'test_preds.csv')))
    per_seed = pd.DataFrame([summarize_run(d, n_boot=0) for d in seed_dirs])
    tests = [pd.read_csv(f, dtype={'image_id': str}) for d in seed_dirs
             for f in sorted(glob.glob(os.path.join(d, 'fold*', 'test_preds.csv')))]
    ens = ensemble(tests)
    ens.to_csv(os.path.join(config_dir, 'test_preds_seedensemble.csv'), index=False)
    te = compute_metrics(ens)
    ci = bootstrap_ci(ens, MAIN, n_boot=n_boot) if n_boot else {m: (np.nan, np.nan) for m in MAIN}
    parts = os.path.normpath(config_dir).split(os.sep)
    row = {'encoder': parts[-2], 'model': parts[-1], 'n_seeds': len(seed_dirs), 'n_models': len(tests)}
    for m in MAIN:
        row[f'seedmean_{m}'] = per_seed[f'test_{m}'].mean()
        row[f'seedstd_{m}'] = per_seed[f'test_{m}'].std() if len(per_seed) > 1 else np.nan
        row[f'cv_{m}_mean'] = per_seed[f'cv_{m}'].mean()
        row[f'ens_{m}'] = te[m]
        row[f'ens_{m}_lo'], row[f'ens_{m}_hi'] = ci[m]
    row.update({f'ens_{k}': v for k, v in te.items() if k.startswith('recall_')})
    return row


def main():
    p = argparse.ArgumentParser(description='Summarize MIL runs')
    p.add_argument('runs', nargs='+', help='run folders (each with fold1..fold5)')
    p.add_argument('--out', default=None, help='CSV path for the combined table')
    p.add_argument('--n_boot', type=int, default=1000)
    args = p.parse_args()

    rows = [summarize_run(r, args.n_boot) for r in args.runs]
    table = pd.DataFrame(rows)
    if args.out:
        os.makedirs(os.path.dirname(args.out) or '.', exist_ok=True)
        table.to_csv(args.out, index=False)

    def fmt(r, m):
        return f"{r[f'test_{m}']:.3f} [{r[f'test_{m}_lo']:.3f}-{r[f'test_{m}_hi']:.3f}]"

    print(f"{'run':45s} {'CV bacc':>8s}  {'test bacc [95% CI]':>22s}  {'test F1 [95% CI]':>22s}  {'test AUC [95% CI]':>22s}")
    for r in rows:
        print(f"{r['run'][-45:]:45s} {r['cv_bacc']:8.3f}  {fmt(r, 'bacc'):>22s}  {fmt(r, 'macro_f1'):>22s}  {fmt(r, 'macro_auc'):>22s}")


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /tmp/code/mil/grid.py
"""Run many MIL trainings in one process: every encoder x model x seed x fold.

Each encoder's features are loaded ONCE onto the device and shared by all its runs.
A run is "done" when <out>/<encoder>/<model>/seed<s>/fold<k>/test_preds.csv exists; done runs are
skipped, and finished runs found in --prev roots (earlier Kaggle versions) are copied instead of redone.
No new run starts after --time_budget_h.

    python -m mil.grid --features_root features --splits splits_v2 --out runs \
        --encoders vits16_lunit,plip --models maxpool,abmil --device cuda:0
"""
import argparse
import os
import shutil
import time
from types import SimpleNamespace

import pandas as pd
import torch

from .data import preload_features
from .train import train_fold

DEFAULTS = dict(epochs=50, min_epochs=10, patience=10, lr=2e-4, wd=1e-4, accum=1, max_patches=8000,
                select='loss', class_weights=False, cache=False)


def run_dir(out, enc, model, seed, fold):
    return os.path.join(out, enc, model, f'seed{seed}', f'fold{fold}')


def is_done(d):
    return os.path.exists(os.path.join(d, 'test_preds.csv'))


def all_slide_ids(splits):
    files = [os.path.join(splits, 'test.csv')] + [os.path.join(splits, s, f'{k}.csv') for s in ('train', 'val') for k in range(1, 6)]
    return sorted(set(pd.concat([pd.read_csv(f, dtype={'file_name': str}) for f in files]).file_name))


def main():
    p = argparse.ArgumentParser(description='Grid of MIL trainings')
    p.add_argument('--features_root', required=True, help='folder containing <encoder>/<slide_id>.npy')
    p.add_argument('--splits', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--encoders', required=True, help='comma-separated')
    p.add_argument('--models', default='maxpool,meanpool,abmil,dsmil,clam_sb,transmil')
    p.add_argument('--folds', default='1,2,3,4,5')
    p.add_argument('--seeds', default='0')
    p.add_argument('--device', default='cuda:0' if torch.cuda.is_available() else 'cpu')
    p.add_argument('--time_budget_h', type=float, default=1e9)
    p.add_argument('--prev', default='', help='comma-separated earlier output roots (with <encoder>/... inside)')
    for k, v in DEFAULTS.items():
        if isinstance(v, bool):
            p.add_argument(f'--{k}', action='store_true', default=v)
        else:
            p.add_argument(f'--{k}', type=type(v), default=v)
    a = p.parse_args()

    t0 = time.time()
    encoders, models = a.encoders.split(','), a.models.split(',')
    folds = [int(f) for f in a.folds.split(',')]
    seeds = [int(s) for s in a.seeds.split(',')]
    prev = [r for r in a.prev.split(',') if r]
    ids = all_slide_ids(a.splits)
    device = torch.device(a.device)
    total = len(encoders) * len(models) * len(seeds) * len(folds)
    n = 0

    for enc in encoders:
        todo = [(m, s, f) for m in models for s in seeds for f in folds if not is_done(run_dir(a.out, enc, m, s, f))]
        # reuse finished runs from earlier versions
        for m, s, f in list(todo):
            for root in prev:
                src = run_dir(root, enc, m, s, f)
                if is_done(src):
                    shutil.copytree(src, run_dir(a.out, enc, m, s, f), dirs_exist_ok=True)
                    todo.remove((m, s, f))
                    print(f'[{enc}] copied {m} seed{s} fold{f} from {root}', flush=True)
                    break
        n += (len(models) * len(seeds) * len(folds)) - len(todo)
        if not todo:
            print(f'[{enc}] all runs done', flush=True)
            continue
        if time.time() - t0 > a.time_budget_h * 3600:
            break

        t_load = time.time()
        cache = preload_features(os.path.join(a.features_root, enc), ids, device)
        gb = sum(x.numel() * x.element_size() for x in cache.values()) / 1e9
        print(f'[{enc}] loaded {len(cache)} bags ({gb:.2f} GB) to {device} in {time.time() - t_load:.0f}s', flush=True)

        for m, s, f in todo:
            if time.time() - t0 > a.time_budget_h * 3600:
                print(f'time budget reached ({a.time_budget_h} h): stopping before {enc}/{m}/seed{s}/fold{f}', flush=True)
                break
            t_run = time.time()
            args = SimpleNamespace(**{k: getattr(a, k) for k in DEFAULTS}, features=os.path.join(a.features_root, enc),
                                   splits=a.splits, model=m, seed=s, out=os.path.join(a.out, enc, m, f'seed{s}'))
            train_fold(args, f, device, feature_cache=cache)
            n += 1
            el = time.time() - t0
            print(f'== [{n}/{total}] {enc}/{m}/seed{s}/fold{f} done in {(time.time() - t_run) / 60:.1f} min '
                  f'| elapsed {el / 3600:.2f} h', flush=True)
        else:
            del cache
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            continue
        break

    done = sum(is_done(run_dir(a.out, e, m, s, f)) for e in encoders for m in models for s in seeds for f in folds)
    print(f'GRID FINISHED: {done}/{total} runs complete'
          + ('' if done == total else ' -> run again with this output attached to continue'), flush=True)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /tmp/code/mil/stats.py
"""Paired bootstrap significance tests between configurations on the locked test set.

For every pair (A, B) the SAME resampled set of test slides is scored for both models
(paired bootstrap, 10,000 resamples), giving the difference A - B, its 95% CI and a two-sided p-value.
p-values are Holm-corrected within each family of pre-specified comparisons:

  encoder_vs_vits16   each encoder vs ViT-S/16 Lunit, same MIL model
  model_vs_maxpool    each MIL model vs MaxPool, same encoder
  capstone_claims     ABMIL vs DSMIL and MaxPool vs MeanPool, per encoder (claims of the old paper)
  encoder_overall     each encoder vs ViT-S/16, averaged over the 6 MIL models
  model_overall       each MIL model vs MaxPool, averaged over the 4 encoders

Predictions used per configuration (runs/<encoder>/<model>/): test_preds_seedensemble.csv if present
(all seeds x folds averaged), else seed0/test_preds_ensemble.csv.

    python -m mil.stats runs --out results/significance.csv
"""
import argparse
import os

import numpy as np
import pandas as pd

from .data import CLASSES
from .metrics import PROB_COLS

ENCODERS = ['vits16_lunit', 'ctranspath', 'phikon_v2', 'plip']
MODELS = ['maxpool', 'meanpool', 'abmil', 'dsmil', 'clam_sb', 'transmil']
K = len(CLASSES)


def load_config_preds(config_dir, split='test'):
    """Predictions of one configuration folder (<config_dir>/seed*/fold*/...), as (labels, predicted class, image ids).

    split='test': locked test set; probabilities of all seeds x folds averaged per slide
                  (uses test_preds_seedensemble.csv when present).
    split='cv'  : out-of-fold validation predictions of the 430 non-test slides, averaged over seeds.
                  (Each fold's checkpoint was chosen on that same fold, so CV is slightly optimistic, equally for all configs.)
    """
    import glob
    if split == 'cv':
        files = glob.glob(os.path.join(config_dir, 'seed*', 'fold*', 'val_preds.csv'))
    else:
        ens = os.path.join(config_dir, 'test_preds_seedensemble.csv')
        files = [ens] if os.path.exists(ens) else glob.glob(os.path.join(config_dir, 'seed*', 'fold*', 'test_preds.csv'))
    if not files:
        return None
    d = pd.concat([pd.read_csv(f, dtype={'image_id': str}) for f in files])
    d = d.groupby('image_id', as_index=False).agg({'label': 'first', **{c: 'mean' for c in PROB_COLS}})
    d = d.sort_values('image_id').reset_index(drop=True)
    return d['label'].to_numpy(), d[PROB_COLS].to_numpy().argmax(1), d['image_id'].to_numpy()


def load_preds(runs, enc, model, split='test'):
    return load_config_preds(os.path.join(runs, enc, model), split)


def paired_test(y, pred_a, pred_b, metric='bacc', n_boot=10000, seed=0):
    """Paired bootstrap of metric(A) - metric(B) on the same slides: (observed diff, ci_lo, ci_hi, two-sided p)."""
    n = len(y)
    idx = np.random.default_rng(seed).integers(0, n, (n_boot, n))
    full = np.arange(n)[None]
    diff = boot_scores(y, pred_a, idx, metric) - boot_scores(y, pred_b, idx, metric)
    obs = (boot_scores(y, pred_a, full, metric) - boot_scores(y, pred_b, full, metric))[0]
    p = min(1.0, 2 * min((diff <= 0).mean(), (diff >= 0).mean()) + 1 / n_boot)
    return obs, np.percentile(diff, 2.5), np.percentile(diff, 97.5), p


def boot_scores(y, pred, idx, metric):
    """Metric for every bootstrap resample at once. idx: (B, n) -> (B,)"""
    yi, pi = y[idx], pred[idx]
    tp = np.stack([((yi == c) & (pi == c)).sum(1) for c in range(K)], 1).astype(float)   # (B, K)
    support = np.stack([(yi == c).sum(1) for c in range(K)], 1).astype(float)
    predicted = np.stack([(pi == c).sum(1) for c in range(K)], 1).astype(float)
    with np.errstate(invalid='ignore', divide='ignore'):
        recall = tp / support
        if metric == 'bacc':
            return np.nanmean(recall, 1)
        precision = np.where(predicted > 0, tp / predicted, 0.0)
        f1 = np.where(precision + recall > 0, 2 * precision * recall / (precision + recall), 0.0)
        return np.nanmean(np.where(support > 0, f1, np.nan), 1)


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def run(runs, n_boot=10000, seed=0, metrics=('bacc', 'macro_f1'), split='test'):
    preds, ids = {}, None
    for e in ENCODERS:
        for m in MODELS:
            r = load_preds(runs, e, m, split)
            if r is None:
                continue
            y, p, i = r
            if ids is None:
                ids, y_ref = i, y
            assert (i == ids).all() and (y == y_ref).all(), f'{e}/{m}: test slides differ'
            preds[(e, m)] = p
    n = len(ids)
    idx = np.random.default_rng(seed).integers(0, n, (n_boot, n))       # shared resamples (paired)
    full = np.arange(n)[None]

    def score(keys, metric, which):
        """mean metric over one or more configs, per resample (which=idx) or on the full set."""
        return np.mean([boot_scores(y_ref, preds[k], which, metric) for k in keys], 0)

    def test(family, label_a, keys_a, label_b, keys_b, metric):
        keys_a, keys_b = [k for k in keys_a if k in preds], [k for k in keys_b if k in preds]
        if not keys_a or not keys_b:
            return None
        diff = score(keys_a, metric, idx) - score(keys_b, metric, idx)
        obs = (score(keys_a, metric, full) - score(keys_b, metric, full))[0]
        p = min(1.0, 2 * min((diff <= 0).mean(), (diff >= 0).mean()) + 1 / n_boot)
        return dict(family=family, metric=metric, a=label_a, b=label_b, diff=obs,
                    diff_lo=np.percentile(diff, 2.5), diff_hi=np.percentile(diff, 97.5), p=p)

    rows = []
    for metric in metrics:
        for m in MODELS:
            for e in ENCODERS[1:]:
                rows.append(test('encoder_vs_vits16', f'{e}+{m}', [(e, m)], f'vits16_lunit+{m}', [('vits16_lunit', m)], metric))
        for e in ENCODERS:
            for m in MODELS[1:]:
                rows.append(test('model_vs_maxpool', f'{e}+{m}', [(e, m)], f'{e}+maxpool', [(e, 'maxpool')], metric))
            rows.append(test('capstone_claims', f'{e}+abmil', [(e, 'abmil')], f'{e}+dsmil', [(e, 'dsmil')], metric))
            rows.append(test('capstone_claims', f'{e}+maxpool', [(e, 'maxpool')], f'{e}+meanpool', [(e, 'meanpool')], metric))
        for e in ENCODERS[1:]:
            rows.append(test('encoder_overall', e, [(e, m) for m in MODELS], 'vits16_lunit', [('vits16_lunit', m) for m in MODELS], metric))
        for m in MODELS[1:]:
            rows.append(test('model_overall', m, [(e, m) for e in ENCODERS], 'maxpool', [(e, 'maxpool') for e in ENCODERS], metric))

    df = pd.DataFrame([r for r in rows if r])
    df['p_holm'] = np.nan
    for (fam, metric), g in df.groupby(['family', 'metric']):
        df.loc[g.index, 'p_holm'] = holm(g.p.to_numpy())
    df['significant'] = df.p_holm < 0.05
    df.insert(0, 'split', split)
    return df, n


def main():
    ap = argparse.ArgumentParser(description='Paired bootstrap significance tests on the locked test set')
    ap.add_argument('runs', help='runs/ folder with <encoder>/<model>/...')
    ap.add_argument('--out', default=None)
    ap.add_argument('--n_boot', type=int, default=10000)
    ap.add_argument('--split', choices=['test', 'cv'], default='test')
    a = ap.parse_args()
    df, n = run(a.runs, a.n_boot, split=a.split)
    if a.out:
        os.makedirs(os.path.dirname(a.out) or '.', exist_ok=True)
        df.to_csv(a.out, index=False)
    pd.set_option('display.width', 200)
    show = df[df.metric == 'bacc'].copy()
    for c in ['diff', 'diff_lo', 'diff_hi']:
        show[c] = show[c].round(3)
    show['p'] = show.p.round(4); show['p_holm'] = show.p_holm.round(4)
    print(f'[{a.split}] paired bootstrap on {n} slides, {a.n_boot} resamples, Holm-corrected within each family (balanced accuracy)')
    for fam, g in show.groupby('family', sort=False):
        print(f'\n== {fam}\n' + g[['a', 'b', 'diff', 'diff_lo', 'diff_hi', 'p', 'p_holm', 'significant']].to_string(index=False))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /tmp/code/splits_v2/assignment.csv
file_name,label,is_tma,fold
10077,1,0,4
10143,2,0,0
1020,2,0,4
10246,1,0,5
10252,3,0,1
10469,1,0,1
10548,0,0,1
10642,2,0,3
1080,2,0,1
10800,1,0,5
10896,2,0,4
1101,2,0,0
11263,0,0,2
11417,2,0,3
11431,1,0,5
11557,3,0,0
11559,2,0,3
12159,2,0,2
12222,0,0,5
12244,3,0,1
12442,0,0,5
1252,2,0,4
12522,1,0,2
1289,2,0,0
12902,1,0,4
1295,2,0,4
13364,2,0,0
13387,3,0,0
13526,0,0,3
13568,3,1,0
13987,0,0,4
14039,2,0,5
14051,0,0,5
14127,2,0,1
14312,4,0,2
14401,1,0,4
14424,1,0,0
14532,4,0,2
14542,2,0,3
14617,1,0,5
15139,2,0,0
15188,1,0,2
15209,2,0,2
15221,2,0,1
15231,0,0,0
15293,2,0,5
15470,0,0,0
15486,0,0,2
15583,3,0,2
15671,1,0,4
15742,1,0,2
15871,2,0,1
15912,1,0,0
16042,2,0,2
16064,1,0,1
16209,2,0,4
16325,0,0,0
16494,2,0,4
1660,0,0,4
1666,2,0,0
16876,0,0,3
16986,2,0,0
17067,0,0,1
17174,2,0,5
17291,0,0,4
17365,2,0,5
17416,2,0,2
17487,1,0,4
17637,2,1,2
17738,1,0,5
1774,2,0,4
17854,2,0,5
18014,0,0,2
18138,1,0,2
18196,2,0,2
18547,2,0,2
18568,2,0,3
18607,1,0,0
18810,1,0,0
18813,0,0,2
18896,2,0,3
18914,2,0,4
18981,2,0,0
19030,1,0,0
19157,2,0,5
1925,2,0,2
19255,2,0,1
1943,0,0,0
19512,2,0,1
1952,0,0,0
19569,1,0,5
20205,0,0,0
20312,2,0,4
20316,2,0,1
20329,4,0,1
20463,2,0,4
20670,2,0,4
20858,2,0,2
20882,1,0,0
2097,2,0,5
21020,4,1,1
21232,1,0,3
21260,3,0,0
21303,3,0,4
21373,1,0,3
21432,2,0,1
21445,4,0,5
21910,1,0,3
21929,0,0,5
22155,0,0,5
22221,1,0,0
2227,4,0,0
22290,0,0,0
22425,0,0,1
22489,0,0,4
22654,2,0,2
22740,1,0,4
22924,1,0,3
23523,4,0,0
23629,1,0,0
23796,3,0,3
2391,3,0,0
24023,1,0,3
24507,2,0,4
24563,0,0,2
24617,2,0,0
24759,2,0,4
24991,2,0,5
25256,3,0,5
25331,4,0,4
25561,2,0,2
25604,1,0,4
25792,2,0,2
25923,0,0,3
25928,0,0,0
26025,3,0,4
26124,2,0,4
26190,2,0,5
26219,0,0,1
26533,0,0,1
26603,2,0,4
26644,2,0,0
2666,1,0,5
26862,3,0,1
26939,1,0,0
26950,0,0,3
2706,2,0,5
27245,2,0,2
27249,2,0,1
27315,0,0,4
27739,2,0,0
27747,2,0,4
27851,1,0,0
27950,2,0,4
28028,2,0,0
28066,1,0,3
281,3,0,5
28121,2,0,2
28393,2,0,1
28519,2,0,0
28562,4,0,3
286,1,0,2
28603,0,0,1
28736,1,0,5
28821,2,0,4
28922,2,0,1
2906,1,0,4
29084,3,1,5
29147,2,0,1
29200,2,0,1
29240,2,0,0
29331,2,0,3
29615,0,0,1
29888,2,0,2
29904,2,0,3
29915,0,0,5
30203,0,0,1
30272,0,0,1
30369,0,0,5
30508,2,0,5
30515,1,0,2
30539,2,0,3
3055,2,0,5
30712,1,0,4
30738,2,0,2
30792,1,0,3
30794,2,0,3
3084,4,0,2
30868,4,0,5
3092,3,0,4
3098,2,0,5
30986,4,0,2
31033,0,0,2
31297,4,0,1
31300,3,0,0
31333,2,0,0
31383,1,0,1
31473,2,0,0
31594,1,1,3
31793,3,0,5
3191,0,0,2
32032,0,0,1
32035,2,0,4
32042,1,0,1
32112,2,0,5
32192,2,0,1
3222,1,0,5
32432,1,0,2
32596,2,0,4
32636,1,0,5
3264,2,0,3
33708,3,0,1
33839,3,0,2
33976,3,0,0
33984,1,0,1
34247,2,0,1
34277,3,0,4
34508,4,0,1
34649,2,0,1
34688,3,0,4
34690,1,0,4
34720,2,0,1
34822,1,0,3
34845,2,0,5
3511,1,0,0
35197,0,0,5
35239,2,0,1
35565,4,1,5
35592,2,0,5
35652,2,0,0
35754,1,0,1
35792,4,0,4
35909,1,0,0
35953,1,0,1
35998,2,0,5
36008,2,0,2
36063,0,0,1
36204,2,0,2
36302,0,1,5
36499,2,0,0
36513,2,0,0
36583,3,1,3
36639,2,0,1
36678,4,0,2
3672,3,0,2
36783,4,1,2
37190,4,0,5
37307,4,0,4
37367,2,0,5
37385,3,1,4
37655,2,0,1
38018,2,0,1
38019,4,0,4
38041,2,0,3
38048,2,0,5
38097,2,0,2
38118,1,0,1
38349,0,0,0
38366,3,0,0
38479,1,0,0
38535,2,0,1
38585,3,0,5
38647,1,0,3
38669,0,0,4
38687,0,0,3
3881,2,0,2
38849,3,0,1
38901,1,0,5
38959,3,0,3
39144,2,0,1
39146,2,0,3
39172,2,0,0
39208,4,0,0
39252,4,0,2
39255,2,0,2
39258,3,0,2
39269,1,0,4
39297,2,0,2
39365,2,0,5
39425,2,0,0
39466,3,0,5
39728,2,0,4
39872,4,0,3
39880,0,0,4
39893,2,0,4
3997,4,0,0
39990,2,0,0
4,2,0,4
40079,1,0,2
40124,1,0,2
40129,0,0,2
40503,1,0,5
40639,2,0,0
40864,3,1,2
40888,0,0,5
41099,3,0,2
4134,4,1,0
41361,2,0,4
41368,1,1,5
41586,0,1,1
41801,2,0,4
4211,2,0,5
42125,0,0,4
42260,4,0,5
42296,0,0,0
42549,0,0,0
42857,0,1,0
431,2,0,0
43280,1,0,0
43390,2,0,0
43432,2,0,1
43671,2,0,3
43796,0,0,3
43815,2,0,3
43875,2,0,3
43998,1,0,0
44232,1,0,3
44283,0,0,2
44432,2,0,2
44530,4,0,1
44581,0,0,0
44603,0,1,2
44700,1,0,5
44804,1,0,4
44962,1,0,2
44976,0,0,2
45104,2,0,5
45185,2,0,2
45254,1,0,1
45578,1,0,1
45630,1,0,3
45725,0,0,5
45990,0,0,3
4608,1,0,2
46139,2,0,0
46172,2,0,0
46435,2,0,0
46444,1,0,4
46469,2,0,3
46543,0,0,2
46688,1,0,3
46736,2,0,2
46769,0,0,1
46793,2,0,0
46815,4,0,3
47020,2,0,5
47035,1,1,0
47105,1,0,2
47431,4,0,5
47837,2,0,5
47911,2,0,1
47960,0,0,3
4797,4,0,3
47984,2,0,0
4827,0,0,5
48502,4,0,4
48506,0,0,2
48550,4,0,4
48734,1,1,4
4877,0,0,2
48861,2,0,4
48973,2,0,2
49281,2,0,1
49587,4,0,0
4963,2,0,4
49872,3,0,4
49942,1,0,3
49995,1,0,3
50048,1,0,0
5015,2,0,5
50246,1,0,3
50304,1,0,0
50589,2,0,3
50712,2,0,3
50878,2,0,3
50932,2,1,4
50962,1,0,1
51021,2,0,2
51032,1,0,5
51128,1,0,0
5114,2,0,2
51215,1,0,2
51346,0,0,0
51499,1,0,3
51679,1,0,1
51832,2,0,3
51893,4,0,4
52108,1,0,4
52259,0,0,4
52275,2,0,5
52308,2,0,0
52375,0,0,4
52420,0,0,0
52438,2,0,2
52461,0,0,4
5251,2,0,1
52612,3,0,0
5264,2,0,3
5265,1,0,0
52752,1,0,4
52784,2,0,3
52836,1,0,5
52846,1,0,1
52931,0,0,1
53059,2,0,2
5307,2,0,4
53098,1,0,0
53377,0,0,3
53402,1,0,4
53655,1,1,1
53688,2,0,1
53859,1,0,5
53900,1,0,3
54007,2,0,1
54408,1,0,0
54473,3,0,4
54506,2,0,4
5456,4,0,3
54590,2,0,0
54825,2,0,4
54928,0,0,3
54949,2,0,0
54990,2,0,3
55279,1,0,1
55281,2,0,2
55287,0,0,5
55876,1,0,5
56117,4,0,0
56221,2,0,5
56351,4,0,2
56500,3,0,3
56799,4,0,3
56843,2,0,3
56861,0,0,1
56875,1,0,0
56947,0,0,3
56993,4,0,0
57100,2,0,2
57162,3,0,3
57265,4,0,5
57468,2,0,5
57598,0,0,0
57696,0,1,3
57711,1,0,2
5851,2,0,4
5852,3,0,1
58895,2,0,0
58947,0,0,3
58974,0,0,2
59002,1,0,3
59031,2,0,0
59383,2,0,2
59515,0,0,4
59589,2,0,3
5970,0,0,3
59760,1,0,2
59900,1,0,1
5992,2,0,5
60058,2,0,3
60287,2,0,0
60685,0,0,0
60928,1,0,2
60936,1,0,1
60988,0,0,5
61033,2,0,3
61089,2,0,1
61100,3,0,3
61280,2,0,5
61320,1,0,1
6140,2,0,5
61424,2,0,4
61493,2,0,3
61689,2,0,3
6175,2,0,3
61797,2,1,0
61823,1,0,4
61852,2,0,5
61961,0,0,0
62476,4,0,0
62641,2,0,0
6281,1,0,2
62828,0,0,1
63015,0,0,0
63121,1,0,3
63165,0,0,0
63289,0,0,5
63298,2,0,0
63367,1,0,5
63429,1,0,5
6359,0,0,4
6363,1,0,4
63836,1,0,1
63897,2,0,3
63941,2,0,1
64111,2,0,0
64188,2,0,1
6449,0,0,4
64629,2,0,1
64771,1,0,0
64824,0,0,3
64950,2,0,0
65022,3,0,3
65094,4,0,1
65300,2,0,2
65371,2,0,0
65533,2,0,2
6558,2,0,3
6582,4,0,3
66,3,0,1
6793,1,0,2
6843,2,0,2
6898,3,0,5
6951,2,0,3
706,2,0,0
7204,1,0,4
7329,1,0,0
7482,0,0,3
7490,1,0,2
7955,0,0,4
8130,3,0,5
8213,2,0,5
8240,2,0,4
8279,0,0,5
8280,2,1,3
8531,1,0,1
8713,2,0,4
8805,2,0,0
8985,3,0,3
91,2,1,1
9154,2,0,1
9183,3,0,2
9200,4,1,4
9254,4,0,1
9341,2,0,0
9509,2,0,5
9658,0,0,2
9697,2,0,5
970,2,0,1

In [ ]:
# 2 · unpack splits: fold 0 = locked test set, fold k = validation set of CV fold k
import pandas as pd
S = '/tmp/code/splits_v2'
a = pd.read_csv(f'{S}/assignment.csv', dtype={'file_name': str})
os.makedirs(f'{S}/train', exist_ok=True); os.makedirs(f'{S}/val', exist_ok=True)
cols = ['file_name', 'label', 'is_tma']
a[a.fold == 0][cols].to_csv(f'{S}/test.csv', index=False)
for k in range(1, 6):
    a[(a.fold != 0) & (a.fold != k)][cols].to_csv(f'{S}/train/{k}.csv', index=False)
    a[a.fold == k][cols].to_csv(f'{S}/val/{k}.csv', index=False)
assert len(a) == 538 and (a.fold == 0).sum() == 108
print('splits ready:', a.fold.value_counts().sort_index().to_dict())

In [ ]:
# 3 · settings
import glob
SMOKE_TEST    = True       # True = quick check. Set False for the full run (Step B).
ENCODERS      = ['phikon_v2', 'ctranspath', 'vits16_lunit', 'plip']   # 2 GPUs -> [phikon_v2, vits16_lunit] + [ctranspath, plip]
MODELS        = ['maxpool', 'meanpool', 'abmil', 'dsmil', 'clam_sb', 'transmil']
SEEDS         = '0,1,2'    # seed 0 is copied from an attached earlier run if present
TIME_BUDGET_H = 10.0      # no new run starts after this (Kaggle stops sessions at 12 h)
OUT           = '/kaggle/working'

metas = glob.glob('/kaggle/input/**/meta.json', recursive=True)
roots = [os.path.dirname(m) for m in metas if os.path.isdir(os.path.join(os.path.dirname(m), 'features', 'plip'))]
assert roots, 'Week 2 features not attached: Add Input -> Your Work -> Notebooks -> notebook4297f7f927'
FEATURES_ROOT = os.path.join(roots[0], 'features')
# earlier versions of THIS notebook attached as input (Step C): their runs/ folders
from pathlib import Path
PREV = sorted({str(Path(p).parent.parent.parent) for p in glob.glob('/kaggle/input/**/runs/*/*/seed*', recursive=True)})

extra = []
if SMOKE_TEST:
    ENCODERS, MODELS, SEEDS = ['vits16_lunit'], ['maxpool', 'abmil', 'transmil'], '0'
    extra = ['--folds', '1', '--epochs', '5', '--min_epochs', '1', '--patience', '5']

n_gpu = torch.cuda.device_count()
assert n_gpu > 0, 'No GPU: Session options -> Accelerator -> GPU T4 x2'
# split encoders over the GPUs so both get similar work
groups = [ENCODERS[i::n_gpu] for i in range(n_gpu)] if len(ENCODERS) > 1 else [ENCODERS]
print('features:', FEATURES_ROOT, '\nprevious runs:', PREV, '\nGPU plan:', {f'cuda:{i}': g for i, g in enumerate(groups)})

In [ ]:
# 4 · run: one background process per GPU, progress printed every 5 minutes
import subprocess
os.makedirs(f'{OUT}/logs', exist_ok=True)
budget = TIME_BUDGET_H - (time.time() - T0) / 3600
procs = []
for i, encs in enumerate(groups):
    cmd = ['python', '-m', 'mil.grid', '--features_root', FEATURES_ROOT, '--splits', '/tmp/code/splits_v2',
           '--out', f'{OUT}/runs', '--encoders', ','.join(encs), '--models', ','.join(MODELS), '--seeds', SEEDS,
           '--device', f'cuda:{i}', '--time_budget_h', f'{budget:.2f}', '--prev', ','.join(PREV)] + extra
    log = open(f'{OUT}/logs/gpu{i}.log', 'w')
    procs.append((subprocess.Popen(cmd, cwd='/tmp/code', stdout=log, stderr=subprocess.STDOUT), f'{OUT}/logs/gpu{i}.log'))

def tail(path, keys=('==', 'loaded', 'copied', 'GRID', 'budget', 'Error', 'Traceback')):
    lines = [l.rstrip() for l in open(path) if l.startswith(keys) or 'Error' in l]
    return lines[-3:]

while any(p.poll() is None for p, _ in procs):
    time.sleep(10 if SMOKE_TEST else 300)
    print(f'--- {(time.time() - T0) / 3600:.2f} h')
    for p, path in procs:
        for l in tail(path): print(f'  {os.path.basename(path)}: {l}')
for p, path in procs:
    print(f'{os.path.basename(path)} exit code {p.returncode}')
    if p.returncode != 0:
        print(''.join(open(path).readlines()[-30:]))

In [ ]:
# 4b · safety copy: bundle the raw predictions right after training (cell 6 rebuilds it with the results tables)
import zipfile
with zipfile.ZipFile(f'{OUT}/results_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{OUT}/logs/*') + glob.glob(f'{OUT}/runs/**/*.csv', recursive=True) + glob.glob(f'{OUT}/runs/**/*.json', recursive=True):
        z.write(f, os.path.relpath(f, OUT))
print(f'safety bundle: {os.path.getsize(f"{OUT}/results_bundle.zip") / 1e6:.1f} MB')

In [ ]:
# 5 · results on the LOCKED TEST SET
#   per seed: 5-fold ensemble (summary_per_seed.csv)
#   per encoder+model: mean ± std over seeds, and the SEED ENSEMBLE (all seeds x folds averaged) with 95% CI (summary.csv)
import sys; sys.path.insert(0, '/tmp/code')
from pathlib import Path
from mil.evaluate import summarize_run, summarize_config
seed_runs = sorted(d for d in glob.glob(f'{OUT}/runs/*/*/seed*') if glob.glob(f'{d}/fold*/test_preds.csv'))
per_seed = pd.DataFrame([summarize_run(d, n_boot=1000) for d in seed_runs])
per_seed['encoder'] = [Path(r).parts[-3] for r in per_seed.run]; per_seed['model'] = [Path(r).parts[-2] for r in per_seed.run]
per_seed['seed'] = [Path(r).parts[-1] for r in per_seed.run]
configs = sorted({str(Path(d).parent) for d in seed_runs})
res = pd.DataFrame([summarize_config(c, n_boot=1000) for c in configs])
os.makedirs(f'{OUT}/results', exist_ok=True)
per_seed.to_csv(f'{OUT}/results/summary_per_seed.csv', index=False)
res.to_csv(f'{OUT}/results/summary.csv', index=False)

piv = lambda col: res.pivot(index='model', columns='encoder', values=col)
ens = lambda m: res.assign(v=[f"{r[f'ens_{m}']:.3f} [{r[f'ens_{m}_lo']:.2f}-{r[f'ens_{m}_hi']:.2f}]" for _, r in res.iterrows()])
for m, name in [('bacc', 'balanced accuracy'), ('macro_f1', 'macro F1'), ('macro_auc', 'macro AUC')]:
    print(f'\nTEST {name}, seed ensemble [95% CI]  (rows = MIL model, columns = encoder)\n'
          + ens(m).pivot(index='model', columns='encoder', values='v').to_string())
ms = res.assign(v=[f"{r.seedmean_bacc:.3f} ± {r.seedstd_bacc:.3f}" for _, r in res.iterrows()])
print('\nTEST balanced accuracy, mean ± std over seeds\n' + ms.pivot(index='model', columns='encoder', values='v').to_string())
print('\nCV balanced accuracy (mean over seeds; for model selection)\n' + piv('cv_bacc_mean').round(3).to_string())
print(f'\nseeds per config: {res.n_seeds.min()}-{res.n_seeds.max()} | models per config: {res.n_models.min()}-{res.n_models.max()}')

In [ ]:
# 5b · paired bootstrap significance tests (Holm-corrected), on the test set and on the CV predictions
from mil.stats import run as significance
for split in ['test', 'cv']:
    sig, n = significance(f'{OUT}/runs', n_boot=10000, split=split)
    sig.to_csv(f'{OUT}/results/significance_{split}.csv', index=False)
    b = sig[sig.metric == 'bacc']
    print(f'\n[{split}: {n} slides] significant after Holm: {int(b.significant.sum())} of {len(b)} comparisons (balanced accuracy)')
    show = b[b.significant | b.family.str.endswith('overall')][['family', 'a', 'b', 'diff', 'diff_lo', 'diff_hi', 'p_holm', 'significant']]
    print(show.round(3).to_string(index=False))

In [ ]:
# 6 · small download bundle: results/ + every prediction file (no model weights)
import shutil, zipfile
with zipfile.ZipFile(f'{OUT}/results_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{OUT}/results/*') + glob.glob(f'{OUT}/logs/*'):
        z.write(f, os.path.relpath(f, OUT))
    for f in glob.glob(f'{OUT}/runs/**/*.csv', recursive=True) + glob.glob(f'{OUT}/runs/**/*.json', recursive=True):
        z.write(f, os.path.relpath(f, OUT))
print(f'results_bundle.zip: {os.path.getsize(f"{OUT}/results_bundle.zip") / 1e6:.1f} MB (download this one file)')